In [1]:
import plotly.graph_objects as go
import numpy as np
from collections import defaultdict

# =========================================================
# 🎨 ตั้งค่าสีและฟังก์ชันส่วนกลาง
# =========================================================
BASE_COLOR = "#FFF4F4"
EDGE_COLOR = "#A78F8F"

def get_btype(dx, dy, dz):
    """ฟังก์ชันจัดระเบียบชื่อบล็อกมาตรฐาน"""
    dims = sorted([int(dx), int(dy), int(dz)])
    return f"brick_{dims[0]}x{dims[1]}x{dims[2]}"

# =========================================================
# 🪑 1. โต๊ะ / ชั้นวางของอเนกประสงค์ (Table)
# =========================================================
def generate_table(w=32, l=20, h=16):
    blocks_used = []

    # 1. กำหนดขนาดสุทธิและปัดให้เป็นเลขคู่เพื่อให้เลโก้ลงล็อก
    real_w = max(8, int(w))
    real_l = max(8, int(l))
    real_h = max(4, int(h))

    real_w += real_w % 2
    real_l += real_l % 2
    real_h += real_h % 2

    thickness = 2

    # 2. คำนวณจำนวนชั้นวาง (อ้างอิงจาก Base Scale 1 ที่ H=16)
    # ให้ชั้นล่างสุดลอยจากพื้นตามสัดส่วนความสูง (h/4)
    bottom_z = max(2, int(real_h / 4))
    bottom_z -= bottom_z % 2 # ปัดให้เป็นเลขคู่
    top_z = real_h - thickness

    # 🔥 เงื่อนไข: ถ้าเตี้ยมาก (<= 8) ให้มีแค่ 1 ชั้น (ท็อปโต๊ะ)
    if real_h <= 8:
        num_shelves = 1
        z_levels = [top_z]
    else:
        # ทุกๆ ความสูง 16 หน่วย จะเพิ่ม 1 ชั้น
        num_shelves = max(2, int(real_h / 16) + 1)
        if num_shelves == 2:
            z_levels = [bottom_z, top_z]
        else:
            raw_levels = np.linspace(bottom_z, top_z, num_shelves)
            # ปัดเศษให้ตกเลขคู่ทุกชั้น เพื่อไม่ให้เกิดเศษทศนิยมตอนต่อเลโก้
            z_levels = [int(round(z / 2.0)) * 2 for z in raw_levels]

    # 3. คำนวณตำแหน่งเสา (เฉพาะฝั่งทางยาว X-axis)
    x_legs = [0]
    # อ้างอิง Base Scale 1 ที่กว้าง 32: ทุกๆ ความกว้าง 32 จะมีเสาคั่น
    num_x_spans = max(1, int(real_w / 32))
    step_x = (real_w - 2) / num_x_spans

    for i in range(1, num_x_spans):
        pos = int(round(i * step_x / 2.0)) * 2
        x_legs.append(pos)
    x_legs.append(real_w - 2)
    x_legs = sorted(list(set(x_legs)))

    # ฝั่งความกว้าง (Y-axis) ล็อกไว้แค่เสามุม ซ้าย-ขวา ไม่มีเสากลาง
    y_legs = [0, real_l - 2]

    leg_positions = [(x, y) for x in x_legs for y in y_legs]

    # 4. สร้างเสาโต๊ะ (ใช้บล็อกมาตรฐานความสูงสูงสุด 8)
    def pack_legs(start_z, target_h):
        for cx, cy in leg_positions:
            current_z = start_z
            rem_h = target_h
            while rem_h > 0:
                bh = int(min(8, rem_h))
                if bh % 2 != 0: bh += 1 # เผื่อกรณีกันเหนียวให้เป็นเลขคู่
                if bh > rem_h: bh = int(rem_h)

                blocks_used.append({
                    'type': get_btype(2, 2, bh), 'color': BASE_COLOR,
                    'x': cx, 'y': cy, 'z': current_z,
                    'dx': 2, 'dy': 2, 'dz': bh
                })
                current_z += bh
                rem_h -= bh

    # วางเสาตามช่องว่างระหว่างชั้น
    pack_legs(0, z_levels[0])
    for i in range(len(z_levels) - 1):
        pack_legs(z_levels[i] + thickness, z_levels[i+1] - (z_levels[i] + thickness))

    # 5. สร้างแผ่นชั้นวาง (ปูด้วยบล็อกไซส์มาตรฐาน 2x8x8 หรือเล็กกว่า)
    def pack_shelf(z_level):
        # 5.1 วางมุม 4 ด้านและจุดที่ตรงกับเสา
        for cx, cy in leg_positions:
            blocks_used.append({
                'type': get_btype(2, 2, thickness), 'color': BASE_COLOR,
                'x': cx, 'y': cy, 'z': z_level, 'dx': 2, 'dy': 2, 'dz': thickness
            })

        # 5.2 ขอบซ้าย-ขวา (Y-axis)
        for y in range(2, real_l - 2, 8):
            dy = min(8, real_l - 2 - y)
            if dy > 0:
                blocks_used.append({'type': get_btype(2, dy, thickness), 'color': BASE_COLOR, 'x': 0, 'y': y, 'z': z_level, 'dx': 2, 'dy': dy, 'dz': thickness})
                blocks_used.append({'type': get_btype(2, dy, thickness), 'color': BASE_COLOR, 'x': real_w - 2, 'y': y, 'z': z_level, 'dx': 2, 'dy': dy, 'dz': thickness})

        # 5.3 ขอบหน้า-หลัง และ แกนกลาง
        for x in range(2, real_w - 2, 8):
            dx = min(8, real_w - 2 - x)

            if dx > 0:
                # ปูขอบหน้า-หลัง (X-axis)
                blocks_used.append({'type': get_btype(dx, 2, thickness), 'color': BASE_COLOR, 'x': x, 'y': 0, 'z': z_level, 'dx': dx, 'dy': 2, 'dz': thickness})
                blocks_used.append({'type': get_btype(dx, 2, thickness), 'color': BASE_COLOR, 'x': x, 'y': real_l - 2, 'z': z_level, 'dx': dx, 'dy': 2, 'dz': thickness})

                # ถมแผ่นใหญ่ตรงกลาง
                for y in range(2, real_l - 2, 8):
                    dy = min(8, real_l - 2 - y)
                    if dy > 0:
                        blocks_used.append({'type': get_btype(dx, dy, thickness), 'color': BASE_COLOR, 'x': x, 'y': y, 'z': z_level, 'dx': dx, 'dy': dy, 'dz': thickness})

    for zl in z_levels:
        pack_shelf(zl)

    return blocks_used, real_w, real_l, real_h

# =========================================================
# 👟 2. ชั้นวางรองเท้า (Shoe Rack)
# =========================================================
def generate_shoe_rack(w, l, h, has_walls=False):
    blocks_used = []
    real_w = max(40, int(w)); real_w += real_w % 2
    real_l = max(20, int(l)); real_l += real_l % 2
    real_h = max(20, int(h)); real_h += real_h % 2
    thickness = 2; bottom_z = 8; top_z = real_h - thickness

    num_shelves = max(2, int((real_h - bottom_z) / 18) + 1)
    raw_levels = np.linspace(bottom_z, top_z, num_shelves)
    z_levels = [int(round(z / 2.0)) * 2 for z in raw_levels]

    leg_size = 4
    if has_walls:
        x_legs = [0, real_w - leg_size]
        y_legs = [0, real_l - leg_size]
    else:
        x_legs = [2, real_w - leg_size - 2]
        y_legs = [2, real_l - leg_size - 2]

    if real_w >= 100:
        x_legs.insert(1, (real_w // 2) - ((real_w // 2) % 2))

    def pack_block(start_x, end_x, start_y, end_y, start_z, target_h):
        if target_h <= 0 or start_x >= end_x or start_y >= end_y: return
        for z in range(0, target_h, 8):
            dz = min(8, target_h - z)
            if dz % 2 != 0: dz += 1
            if dz > target_h - z: dz = int(target_h - z)
            for x in range(start_x, end_x, 8):
                dx = min(8, end_x - x)
                for y in range(start_y, end_y, 8):
                    dy = min(8, end_y - y)
                    blocks_used.append({'type': get_btype(dx, dy, dz), 'color': BASE_COLOR, 'x': x, 'y': y, 'z': start_z + z, 'dx': dx, 'dy': dy, 'dz': dz})

    for cx in x_legs:
        for cy in y_legs:
            pack_block(cx, cx + leg_size, cy, cy + leg_size, 0, z_levels[0])
            for i in range(len(z_levels) - 1):
                start_z = z_levels[i] + thickness
                target_h = z_levels[i+1] - start_z
                pack_block(cx, cx + leg_size, cy, cy + leg_size, start_z, target_h)

    if has_walls:
        pack_block(0, thickness, 0, real_l, z_levels[0], real_h - z_levels[0])
        pack_block(real_w - thickness, real_w, 0, real_l, z_levels[0], real_h - z_levels[0])
        pack_block(thickness, real_w - thickness, real_l - thickness, real_l, z_levels[0], real_h - z_levels[0])
        for zl in z_levels:
            pack_block(thickness, real_w - thickness, 0, real_l - thickness, zl, thickness)
        pack_block(0, real_w, 0, real_l, real_h, thickness)
        final_h = real_h + thickness
    else:
        for zl in z_levels:
            pack_block(0, real_w, 0, real_l, zl, thickness)
        final_h = real_h

    return blocks_used, real_w, real_l, final_h

# =========================================================
# 🔌 3. กล่องจัดระเบียบสายไฟ (Cable Box)
# =========================================================
def generate_cable_box(w, l, h):
    blocks_used = []
    real_w, real_l, real_h = max(20, int(w)), max(12, int(l)), max(10, int(h))
    real_w += real_w % 2; real_l += real_l % 2; real_h += real_h % 2
    thick = 2

    def pack_block(start_x, end_x, start_y, end_y, start_z, target_h):
        if target_h <= 0 or start_x >= end_x or start_y >= end_y: return
        for z in range(0, target_h, 8):
            dz = min(8, target_h - z); dz += dz % 2; dz = min(dz, int(target_h - z))
            for x in range(start_x, end_x, 8):
                dx = min(8, end_x - x)
                for y in range(start_y, end_y, 8):
                    dy = min(8, end_y - y)
                    blocks_used.append({'type': get_btype(dx, dy, dz), 'color': BASE_COLOR, 'x': x, 'y': y, 'z': start_z + z, 'dx': dx, 'dy': dy, 'dz': dz})

    pack_block(0, real_w, 0, real_l, 0, thick)
    wall_h = real_h - (thick * 2)
    pack_block(0, real_w, 0, thick, thick, wall_h)
    pack_block(0, real_w, real_l - thick, real_l, thick, wall_h)

    mid_y = real_l // 2
    mid_y -= mid_y % 2
    slit_w = 4
    slit_start = mid_y - (slit_w // 2)
    slit_end = mid_y + (slit_w // 2)

    pack_block(0, thick, thick, slit_start, thick, wall_h)
    pack_block(0, thick, slit_end, real_l - thick, thick, wall_h)
    pack_block(real_w - thick, real_w, thick, slit_start, thick, wall_h)
    pack_block(real_w - thick, real_w, slit_end, real_l - thick, thick, wall_h)

    gap_size = 2
    lid_end_y = real_l - thick - gap_size
    pack_block(0, real_w, 0, lid_end_y, real_h - thick, thick)
    pack_block(0, thick, lid_end_y, real_l - thick, real_h - thick, thick)
    pack_block(real_w - thick, real_w, lid_end_y, real_l - thick, real_h - thick, thick)

    return blocks_used, real_w, real_l, real_h

# =========================================================
# 📱 4. แท่นวางโทรศัพท์/แท็บเล็ต (Device Stand)
# =========================================================
def generate_device_stand(w, l, h):
    blocks_used = []
    real_w = max(10, int(w)); real_w += real_w % 2
    real_l = max(10, int(l)); real_l += real_l % 2
    real_h = max(10, int(h)); real_h += real_h % 2
    thick = 2

    def pack_block(start_x, end_x, start_y, end_y, start_z, target_h):
        if target_h <= 0 or start_x >= end_x or start_y >= end_y: return
        for z in range(0, target_h, 8):
            dz = min(8, target_h - z); dz += dz % 2; dz = min(dz, int(target_h - z))
            for x in range(start_x, end_x, 8):
                dx = min(8, end_x - x)
                for y in range(start_y, end_y, 8):
                    dy = min(8, end_y - y)
                    blocks_used.append({'type': get_btype(dx, dy, dz), 'color': BASE_COLOR, 'x': x, 'y': y, 'z': start_z + z, 'dx': dx, 'dy': dy, 'dz': dz})

    side_width = (real_w - 4) // 2
    side_width -= side_width % 2
    if side_width < 2: side_width = 2
    hole_start = side_width
    hole_end = real_w - side_width
    gap_y = 4

    pack_block(0, hole_start, 0, gap_y, 0, thick)
    pack_block(hole_end, real_w, 0, gap_y, 0, thick)
    pack_block(0, real_w, gap_y, real_l, 0, thick)

    pack_block(0, hole_start, 0, 2, thick, 2)
    pack_block(hole_end, real_w, 0, 2, thick, 2)

    steps = (real_l - gap_y) // 2
    if steps < 1: steps = 1
    available_h = real_h - thick
    step_h = available_h // steps
    step_h -= step_h % 2
    if step_h <= 0: step_h = 2

    current_z = thick
    for i in range(steps):
        y_start = gap_y + (i * 2)
        y_end = real_l
        h_step = step_h
        if i == steps - 1:
            h_step = real_h - current_z
            h_step -= h_step % 2
        if h_step > 0 and y_start < y_end:
            pack_block(0, real_w, y_start, y_end, current_z, h_step)
            current_z += h_step

    return blocks_used, real_w, real_l, real_h

# =========================================================
# ✏️ 5. ที่จัดระเบียบเครื่องเขียน (Stationery Organizer)
# =========================================================
def generate_stationery_organizer(w, l, h):
    blocks_used = []
    real_w = max(16, int(w)); real_w += real_w % 2
    real_l = max(12, int(l)); real_l += real_l % 2
    real_h = max(10, int(h)); real_h += real_h % 2
    thick = 2

    def pack_block(start_x, end_x, start_y, end_y, start_z, target_h):
        if target_h <= 0 or start_x >= end_x or start_y >= end_y: return
        for z in range(0, target_h, 8):
            dz = min(8, target_h - z); dz += dz % 2; dz = min(dz, int(target_h - z))
            for x in range(start_x, end_x, 8):
                dx = min(8, end_x - x)
                for y in range(start_y, end_y, 8):
                    dy = min(8, end_y - y)
                    blocks_used.append({'type': get_btype(dx, dy, dz), 'color': BASE_COLOR, 'x': x, 'y': y, 'z': start_z + z, 'dx': dx, 'dy': dy, 'dz': dz})

    pack_block(0, real_w, 0, real_l, 0, thick)

    div_y = (real_l // 2) - ((real_l // 2) % 2)
    div_x = (real_w // 2) - ((real_w // 2) % 2)

    back_h = real_h - thick
    front_h = (real_h // 2) - ((real_h // 2) % 2)
    if front_h < 4: front_h = 4

    pack_block(0, thick, div_y, real_l, thick, back_h)
    pack_block(real_w - thick, real_w, div_y, real_l, thick, back_h)
    pack_block(thick, real_w - thick, real_l - thick, real_l, thick, back_h)
    pack_block(thick, real_w - thick, div_y, div_y + thick, thick, back_h)

    pack_block(0, thick, 0, div_y, thick, front_h)
    pack_block(real_w - thick, real_w, 0, div_y, thick, front_h)
    pack_block(thick, real_w - thick, 0, thick, thick, front_h)
    pack_block(div_x, div_x + thick, thick, div_y, thick, front_h)

    return blocks_used, real_w, real_l, real_h

# =========================================================
# 🎨 ฟังก์ชัน Render 3D อเนกประสงค์
# =========================================================
def render_3d_model(blocks_used, title, width, length, height):
    fig = go.Figure()

    def add_mesh_box(x0, y0, z0, dx, dy, dz, color):
        x = [x0, x0+dx, x0+dx, x0, x0, x0+dx, x0+dx, x0]
        y = [y0, y0, y0+dy, y0+dy, y0, y0, y0+dy, y0+dy]
        z = [z0, z0, z0, z0, z0+dz, z0+dz, z0+dz, z0+dz]
        i = [7, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2]
        j = [3, 4, 1, 2, 5, 6, 5, 2, 0, 1, 6, 3]
        k = [0, 7, 2, 3, 6, 7, 1, 1, 5, 5, 7, 6]
        fig.add_trace(go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color, opacity=1.0, flatshading=True))

        edges_x, edges_y, edges_z = [], [], []
        for start, end in [(0,1), (1,2), (2,3), (3,0), (4,5), (5,6), (6,7), (7,4), (0,4), (1,5), (2,6), (3,7)]:
            edges_x.extend([x[start], x[end], None])
            edges_y.extend([y[start], y[end], None])
            edges_z.extend([z[start], z[end], None])
        fig.add_trace(go.Scatter3d(x=edges_x, y=edges_y, z=edges_z, mode='lines', line=dict(color=EDGE_COLOR, width=2), hoverinfo='none', showlegend=False))

    type_counter = defaultdict(int)
    for b in blocks_used:
        add_mesh_box(b['x'], b['y'], b['z'], b['dx'], b['dy'], b['dz'], b['color'])
        type_counter[b['type']] += 1

    fig.update_layout(
        title=f"{title} | Size: {width}W x {length}D x {height}H (cm)",
        scene=dict(
            aspectmode='data',
            xaxis=dict(showbackground=False, visible=True, title='Width (X)'),
            yaxis=dict(showbackground=False, visible=True, title='Depth (Y)'),
            zaxis=dict(showbackground=False, visible=True, title='Height (Z)')
        ),
        paper_bgcolor='white', plot_bgcolor='white', showlegend=False
    )

    print(f"\n===== 📋 BILL OF MATERIALS: {title.upper()} =====")
    total_pieces = 0
    for btype, count in sorted(type_counter.items(), reverse=True):
        print(f"📦 {btype} \t-> \t{count} pieces")
        total_pieces += count
    print(f"✅ Total blocks used: {total_pieces} pieces\n")
    fig.show()

# =========================================================
# 🎮 ตัวควบคุมหลัก (Master Controller)
# =========================================================
def build_furniture(item_type, **kwargs):
    item_type = item_type.lower()

    if item_type == 'table':
        w = kwargs.get('w', 32); l = kwargs.get('l', 20); h = kwargs.get('h', 16)
        blocks_data, w, l, h = generate_table(w, l, h)
        render_3d_model(blocks_data, "Smart Table", w, l, h)

    elif item_type == 'shoe_rack':
        w = kwargs.get('w', 80); l = kwargs.get('l', 32); h = kwargs.get('h', 96)
        has_walls = kwargs.get('has_walls', False)
        blocks_data, w, l, h = generate_shoe_rack(w, l, h, has_walls)
        wall_text = "Cabinet (with Walls)" if has_walls else "Open Rack"
        render_3d_model(blocks_data, f"{wall_text} Shoe Rack", w, l, h)

    elif item_type == 'cable_box':
        w = kwargs.get('w', 32); l = kwargs.get('l', 14); h = kwargs.get('h', 14)
        blocks_data, w, l, h = generate_cable_box(w, l, h)
        render_3d_model(blocks_data, "Cable Organizer Box", w, l, h)

    elif item_type == 'device_stand':
        w = kwargs.get('w', 10); l = kwargs.get('l', 12); h = kwargs.get('h', 14)
        blocks_data, w, l, h = generate_device_stand(w, l, h)
        render_3d_model(blocks_data, "Phone/Tablet Stand", w, l, h)

    elif item_type == 'stationery':
        w = kwargs.get('w', 20); l = kwargs.get('l', 16); h = kwargs.get('h', 14)
        blocks_data, w, l, h = generate_stationery_organizer(w, l, h)
        render_3d_model(blocks_data, "Stationery Organizer", w, l, h)

    else:
        print("❌ ไม่รู้จักเฟอร์นิเจอร์ชนิดนี้!")

# =========================================================
# 🚀 จุดทดสอบรันการทำงาน (เลือกคอมเมนต์/เปิดใช้งานได้เลย)
# =========================================================

# 1. ทดสอบโต๊ะ (ปรับความสูงเท่าไหร่มันก็คือชั้นวางของแหละ!)
build_furniture('table', w=32, l=20, h=32)

# 2. ชั้นวางรองเท้า (แบบมีผนัง)
# build_furniture('shoe_rack', w=60, l=40, h=40, has_walls=False)

# 3. กล่องจัดระเบียบสายไฟ
# build_furniture('cable_box', w=70, l=14, h=14)

# 4. แท่นวางโทรศัพท์/แท็บเล็ต
# build_furniture('device_stand', w=10, l=12, h=14)

# 5. ที่จัดระเบียบเครื่องเขียน
# build_furniture('stationery', w=20, l=16, h=14)


===== 📋 BILL OF MATERIALS: SMART TABLE =====
📦 brick_2x8x8 	-> 	18 pieces
📦 brick_2x4x8 	-> 	6 pieces
📦 brick_2x2x8 	-> 	42 pieces
📦 brick_2x2x4 	-> 	6 pieces
📦 brick_2x2x2 	-> 	16 pieces
✅ Total blocks used: 88 pieces



In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from skimage import color
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ==========================================
# ⚙️ CONFIGURATION & SETTINGS
# ==========================================
TARGET_HEX_INPUT = "#FF34B3"    # 🎯 สีเป้าหมาย (เปลี่ยนได้ตรงนี้)
TOTAL_MASS_GRAMS = 100.0        # น้ำหนักชิ้นงานรวมที่ต้องการ (กรัม)
CAP_WEIGHT_GRAMS = 2.5          # น้ำหนักต่อฝา (กรัม)
PIGMENT_STRENGTH_FACTOR = 5.0   # แม่สีเข้มข้นกว่าสีฝาขวดกี่เท่า

# 🎲 Inventory Simulation Settings
N_DRAWS = 5000                  # ← เปลี่ยนได้: จำนวนครั้งที่สุ่มหยิบฝา
RANDOM_SEED = 42                # ← เปลี่ยนได้: seed สำหรับ reproducibility
INVENTORY_SIZE = 60             # จำนวนสีใกล้เคียงที่ดึงมาใช้ใน optimization

# ==========================================
# 🧪 PHYSICS & COLOR SCIENCE (KUBELKA-MUNK)
# ==========================================

def hex_to_rgb_norm(hex_str):
    hex_str = hex_str.lstrip('#')
    return np.array([int(hex_str[i:i+2], 16) for i in (0, 2, 4)]) / 255.0

def rgb_to_hex(rgb_norm):
    rgb = np.clip(rgb_norm * 255, 0, 255).astype(int)
    return '#{:02x}{:02x}{:02x}'.format(rgb[0], rgb[1], rgb[2])

def rgb_to_ks(rgb_norm):
    rgb_lin = np.where(rgb_norm > 0.04045,
                       ((rgb_norm + 0.055) / 1.055) ** 2.4,
                       rgb_norm / 12.92)
    R = np.clip(rgb_lin, 0.01, 0.99)
    return (1.0 - R)**2 / (2.0 * R)

def ks_to_rgb(ks):
    R = 1.0 + ks - np.sqrt(ks**2 + 2.0 * ks)
    rgb_norm = np.where(R > 0.0031308,
                        1.055 * (R ** (1/2.4)) - 0.055,
                        12.92 * R)
    return np.clip(rgb_norm, 0, 1)

def calculate_delta_e(rgb1, rgb2):
    c1 = color.rgb2lab(rgb1.reshape(1, 1, 3))
    c2 = color.rgb2lab(rgb2.reshape(1, 1, 3))
    return color.deltaE_ciede2000(c1[0, 0], c2[0, 0])

# ==========================================
# 📂 DATA LOADING
# ==========================================
try:
    df = pd.read_csv("BRICKIT_palette_hex_optionB_full_sorted_rgb.xlsx - palette.csv")
    print(f"✅ Loaded {len(df)} colors from database.")
except FileNotFoundError:
    print("❌ CSV File not found.")
    exit()

# ==========================================
# 🎲 SIMULATE INVENTORY (สุ่มหยิบฝา)
# ==========================================
print(f"\n🎲 Simulating inventory: drawing {N_DRAWS:,} caps randomly...")

np.random.seed(RANDOM_SEED)

# สุ่มหยิบ N_DRAWS ครั้งจากฝาทั้งหมด (หยิบแล้วใส่คืน)
drawn_indices = np.random.randint(0, len(df), size=N_DRAWS)
drawn_caps = df.iloc[drawn_indices].copy()

# นับจำนวนฝาแต่ละสีที่หยิบได้
inventory = (drawn_caps
             .groupby(['color_en', 'color_th', 'hex', 'r', 'g', 'b', 'group'])
             .size()
             .reset_index(name='stock_caps')
             .sort_values('stock_caps', ascending=False)
             .reset_index(drop=True))

inventory['stock_mass'] = inventory['stock_caps'] * CAP_WEIGHT_GRAMS

print(f"✅ Got {len(inventory)} unique colors from {N_DRAWS:,} draws")
print(f"\n📊 Summary by color group:")
print(inventory.groupby('group')['stock_caps'].sum().sort_values(ascending=False).to_string())

# ==========================================
# 🎯 SETUP TARGET
# ==========================================
target_rgb = hex_to_rgb_norm(TARGET_HEX_INPUT)
target_ks = rgb_to_ks(target_rgb)

# Pre-calculate K/S for inventory
inv_rgb = inventory[['r', 'g', 'b']].values / 255.0
inv_ks = np.array([rgb_to_ks(c) for c in inv_rgb])

# ==========================================
# 🧠 MODEL 1: OPTIMIZE CAP MIXING
# ==========================================
print(f"\n🔄 Running Model 1: Cap Mixing...")

# เลือก top N สีใกล้เคียงกับ target มากที่สุด
inv_lab = color.rgb2lab(inv_rgb.reshape(-1, 1, 3)).reshape(-1, 3)
target_lab = color.rgb2lab(target_rgb.reshape(1, 1, 3)).reshape(1, 3)
distances = np.linalg.norm(inv_lab - target_lab, axis=1)

num_candidates = min(INVENTORY_SIZE, len(inventory))
candidate_indices = np.argsort(distances)[:num_candidates]
subset_ks = inv_ks[candidate_indices]
subset_inventory = inventory.iloc[candidate_indices].copy()
subset_stock_mass = subset_inventory['stock_mass'].values

def objective_m1(weights):
    if np.sum(weights) == 0: return 100.0
    w_norm = weights / np.sum(weights)
    mix_ks = np.dot(w_norm, subset_ks)
    mix_rgb = ks_to_rgb(mix_ks)
    return calculate_delta_e(mix_rgb, target_rgb)

x0 = np.zeros(num_candidates)
x0[0] = 1.0

# Bounds: ถ้า stock ไม่พอ → ใช้เท่าที่มี (stock_mass เป็น max)
bounds = []
for stock_mass in subset_stock_mass:
    max_ratio = min(1.0, stock_mass / TOTAL_MASS_GRAMS)
    bounds.append((0.0, max_ratio))

constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1.0})

res_m1 = minimize(objective_m1, x0, method='SLSQP',
                  bounds=bounds, constraints=constraints,
                  options={'maxiter': 1000, 'ftol': 1e-9})

m1_ratios = res_m1.x
m1_mix_ks = np.dot(m1_ratios, subset_ks)
m1_rgb = ks_to_rgb(m1_mix_ks)
m1_hex = rgb_to_hex(m1_rgb)
m1_de = calculate_delta_e(m1_rgb, target_rgb)

print(f"   Model 1 Result: ΔE = {m1_de:.2f}")

# ==========================================
# 🎨 MODEL 2: PIGMENT ADJUSTMENT
# ==========================================
print("🔄 Running Model 2: Pigment Correction...")

pigments = {
    'Cyan':    '#00FFFF',
    'Magenta': '#FF00FF',
    'Yellow':  '#FFFF00',
    'Black':   '#000000',
    'White':   '#FFFFFF'
}
pigment_names = list(pigments.keys())
pigment_rgbs = np.array([hex_to_rgb_norm(pigments[k]) for k in pigment_names])
pigment_ks = np.array([rgb_to_ks(c) for c in pigment_rgbs])

def objective_m2(added_pigments):
    base_mass = TOTAL_MASS_GRAMS
    base_ks = m1_mix_ks
    pigment_masses = added_pigments
    total_pigment_mass = np.sum(pigment_masses)
    total_optical_weight = base_mass + (total_pigment_mass * PIGMENT_STRENGTH_FACTOR)
    weighted_base_ks = base_ks * base_mass
    weighted_pigment_ks = np.dot(pigment_masses * PIGMENT_STRENGTH_FACTOR, pigment_ks)
    final_ks = (weighted_base_ks + weighted_pigment_ks) / total_optical_weight
    final_rgb = ks_to_rgb(final_ks)
    penalty = total_pigment_mass * 0.1
    return calculate_delta_e(final_rgb, target_rgb) + penalty

x0_p = np.zeros(len(pigments))
bounds_p = [(0.0, 10.0) for _ in pigments]
res_m2 = minimize(objective_m2, x0_p, method='L-BFGS-B', bounds=bounds_p)

m2_added_grams = res_m2.x
final_optical_weight = TOTAL_MASS_GRAMS + np.sum(m2_added_grams * PIGMENT_STRENGTH_FACTOR)
m2_mix_ks = ((m1_mix_ks * TOTAL_MASS_GRAMS) +
             np.dot(m2_added_grams * PIGMENT_STRENGTH_FACTOR, pigment_ks)) / final_optical_weight
m2_rgb = ks_to_rgb(m2_mix_ks)
m2_hex = rgb_to_hex(m2_rgb)
m2_de = calculate_delta_e(m2_rgb, target_rgb)

print(f"   Model 2 Result: ΔE = {m2_de:.2f}")

# ==========================================
# 📊 VISUALIZATION
# ==========================================
def get_de_comment(de):
    if de < 1.0: return "Perceptually Perfect"
    if de < 2.5: return "Good Match"
    if de < 5.0: return "Acceptable"
    return "Visible Difference"

fig, ax = plt.subplots(1, 3, figsize=(15, 6))
fig.suptitle(
    f"Subtractive Color Mixing (Kubelka-Munk)\n"
    f"Target: {TARGET_HEX_INPUT}  |  Inventory from {N_DRAWS:,} random draws",
    fontsize=14
)

ax[0].add_patch(patches.Rectangle((0,0), 1, 1, color=target_rgb))
ax[0].set_title(f"Target\n{TARGET_HEX_INPUT}", fontsize=13, fontweight='bold')
ax[0].axis('off')

ax[1].add_patch(patches.Rectangle((0,0), 1, 1, color=m1_rgb))
ax[1].set_title(f"Model 1: Caps Only\nΔE: {m1_de:.2f}  ({get_de_comment(m1_de)})", fontsize=13)
ax[1].text(0.5, -0.08, m1_hex, ha='center', transform=ax[1].transAxes, fontsize=11)
ax[1].axis('off')

ax[2].add_patch(patches.Rectangle((0,0), 1, 1, color=m2_rgb))
ax[2].set_title(f"Model 2: Caps + Pigments\nΔE: {m2_de:.2f}  ({get_de_comment(m2_de)})", fontsize=13)
ax[2].text(0.5, -0.08, m2_hex, ha='center', transform=ax[2].transAxes, fontsize=11)
ax[2].axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.85)
plt.savefig("color_mix_result.png", dpi=150, bbox_inches='tight')
plt.show()

# ==========================================
# 📝 TEXT REPORT
# ==========================================
print("\n" + "="*60)
print(f"📝 BEST RECIPE FOR {TARGET_HEX_INPUT}")
print(f"   Total Base: {TOTAL_MASS_GRAMS:.1f}g  |  Drawn from {N_DRAWS:,} caps")
print("="*60)

print(f"\n--- 1. BOTTLE CAP MIX ---")
stock_warnings = []
found_caps = False
for idx, ratio in enumerate(m1_ratios):
    if ratio > 0.01:
        row = subset_inventory.iloc[idx]
        grams_needed = ratio * TOTAL_MASS_GRAMS
        caps_needed = grams_needed / CAP_WEIGHT_GRAMS
        caps_available = row['stock_caps']
        grams_available = row['stock_mass']

        # ตรวจสอบ stock
        if grams_needed > grams_available:
            status = f"⚠️  ต้อง {caps_needed:.1f} ฝา แต่มีแค่ {caps_available} ฝา → ใช้เท่าที่มี"
            stock_warnings.append(row['color_en'])
        else:
            status = f"✅ Stock พอ ({caps_available} ฝาในคลัง)"

        print(f"\n  • {row['color_en']} [{row['color_th']}] ({row['hex']})")
        print(f"    → {ratio*100:.1f}% = {grams_needed:.1f}g = {caps_needed:.1f} ฝา")
        print(f"    {status}")
        found_caps = True

if not found_caps:
    print("  (No suitable caps found)")

print(f"\n--- 2. PIGMENT ADDITIVES ---")
has_pigment = False
for name, grams in zip(pigment_names, m2_added_grams):
    if grams > 0.01:
        print(f"  • + {name}: {grams:.2f} g")
        has_pigment = True
if not has_pigment:
    print("  (No extra pigments needed)")

print(f"\n--- 3. FINAL RESULT ---")
print(f"  ΔE (Caps only):        {m1_de:.2f}  → {get_de_comment(m1_de)}")
print(f"  ΔE (Caps + Pigments):  {m2_de:.2f}  → {get_de_comment(m2_de)}")

if stock_warnings:
    print(f"\n⚠️  Stock ไม่พอสำหรับ: {', '.join(stock_warnings)}")
    print(f"   สูตรนี้ใช้ stock เท่าที่มี ΔE อาจคลาดเคลื่อนจากที่แสดง")

print("-"*60)
print("📌 NOTE: ΔE คือค่าทางทฤษฎี (Kubelka-Munk)")
print(f"   N_DRAWS={N_DRAWS:,} | SEED={RANDOM_SEED} | CAP={CAP_WEIGHT_GRAMS}g")
print("="*60)

❌ CSV File not found.

🎲 Simulating inventory: drawing 5,000 caps randomly...


NameError: name 'df' is not defined